In [1]:
import pandas as pd
import yfinance as yf
from tabulate import tabulate
import time



In [2]:
def get_sp500_tickers():
    """
    Fetches S&P 500 tickers and relevant info from Wikipedia.
    """
    url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
    table = pd.read_html(url)
    df = table[0]
    df['Symbol'] = df['Symbol'].str.replace('.', '-', regex=False)  # For yfinance compatibility
    return df[['Symbol', 'Security', 'GICS Sector']]


In [3]:
def fetch_beta(ticker):
    """
    Fetches the beta value for a given stock ticker using yfinance.
    """
    try:
        stock = yf.Ticker(ticker)
        beta = stock.info.get('beta')
        return beta
    except Exception as e:
        print(f"Failed to get data for {ticker}: {e}")
        return None

In [4]:
def sample_by_beta_range(df, step=0.3, samples_per_range=10):
    """
    Splits the DataFrame into beta intervals and selects a sample of companies from each range.

    Returns a dictionary of DataFrames, each representing a beta basket.
    """
    min_beta = df['Beta'].min()
    max_beta = df['Beta'].max()
    baskets = {}

    current = step * (min_beta // step)
    while current < max_beta:
        upper = current + step
        range_label = f"{current:.1f}–{upper:.1f}"
        mask = df[(df['Beta'] >= current) & (df['Beta'] < upper)]
        sample = mask.sample(n=min(samples_per_range, len(mask)), random_state=42)
        baskets[range_label] = sample.reset_index(drop=True)
        current += step

    return baskets


In [5]:
if __name__ == "__main__":
    # Step 1: Get tickers
    sp500_df = get_sp500_tickers()

    # Step 2: Fetch beta values (with pause to respect rate limits)
    betas = []
    for ticker in sp500_df['Symbol']:
        beta = fetch_beta(ticker)
        betas.append(beta)
        time.sleep(0.5)  # Respect API rate limits

    sp500_df['Beta'] = betas

    # Step 3: Clean and sort by Beta
    sp500_df = sp500_df.dropna(subset=['Beta'])
    sp500_df['Beta'] = sp500_df['Beta'].astype(float)
    sp500_df = sp500_df.sort_values(by='Beta', ascending=False)

    # Step 4: Save full sorted beta list
    sp500_df.to_csv('sp500_beta_by_sector_sorted.csv', index=False)

    # Step 5: Print all sorted companies
    print("\nAll S&P 500 Companies Sorted by Beta:\n")
    print(tabulate(sp500_df, headers='keys', tablefmt='pretty', showindex=False))

    # Step 6: Sample and sort companies into beta baskets
    beta_baskets = sample_by_beta_range(sp500_df, step=0.5, samples_per_range=8)

    # Step 7: Save and print each basket
    for label, basket_df in beta_baskets.items():
        print(f"\nBeta Range {label}:\n")
        print(tabulate(basket_df, headers='keys', tablefmt='pretty', showindex=False))
        basket_df.to_csv(f"sp500_beta_basket_{label.replace('–', '_')}.csv", index=False)


All S&P 500 Companies Sorted by Beta:

+--------+----------------------------------------+------------------------+-------+
| Symbol |                Security                |      GICS Sector       | Beta  |
+--------+----------------------------------------+------------------------+-------+
|  APA   |            APA Corporation             |         Energy         | 2.745 |
|  PLTR  |         Palantir Technologies          | Information Technology | 2.741 |
|  CZR   |         Caesars Entertainment          | Consumer Discretionary | 2.675 |
|  TSLA  |              Tesla, Inc.               | Consumer Discretionary | 2.58  |
|  CCL   |                Carnival                | Consumer Discretionary | 2.422 |
|  NCLH  |     Norwegian Cruise Line Holdings     | Consumer Discretionary | 2.274 |
|  RCL   |         Royal Caribbean Group          | Consumer Discretionary | 2.271 |
|  MRNA  |                Moderna                 |      Health Care       | 2.233 |
|  BLDR  |          Build